# F1 · Optional: the baseline to beat, counting pairs

**Tiny Transformer, episode F1 (optional).** Our tiny transformer scores 1.93 on held-back text. Is that good? A score
means something only beside a **baseline**: the simplest next-character model there is. Here you build it by hand: count which
character follows which in the training text, turn the counts into probabilities, write with them, score them on the same loss
as the transformer, and see why counting longer contexts cannot catch up.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/F1-counting.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook F1-counting.ipynb`.

**Computed here:** every count, probability and score, from Tiny Shakespeare's training text (its first 90%), and the
transformer's own probabilities and score, from its trained numbers (`prompter.json`, step 98,000). **Illustrative:** two small
lines of counts (6, 3, 1 and 3, 1, 0), small enough to work by hand.

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and Sprout's checkpoint) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import base64, collections, hashlib, json, math, os, urllib.request
import numpy as np
import torch

print(torch.__version__)

2.10.0+cu128


**The text, its vocabulary and this episode's code.** Tiny Shakespeare, pinned by its sha256; the first 90% is the training
text, the last 10% is held back, exactly as for the transformer. The **vocabulary** is every distinct character of the text,
sorted (F0), and a character's id is its place in it. The teaching functions are `code/counting.py`.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
train, held_back = text[:int(0.9 * len(text))], text[int(0.9 * len(text)):]
print(f"{len(text):,} characters: {len(train):,} to count, {len(held_back):,} held back")

1,115,394 characters: 1,003,854 to count, 111,540 held back


In [4]:
import math
import torch


def vocabulary(text):
    """Every distinct character of the text, sorted: the model's 65 for Tiny Shakespeare."""
    return sorted(set(text))


def encode(text, vocab):
    """Characters to ids: each character's place in the vocabulary."""
    return [vocab.index(ch) for ch in text]


def decode(ids, vocab):
    """Ids back to characters."""
    return "".join(vocab[i] for i in ids)


def count_pairs(ids, V):
    """counts[a][b]: how often id b came right after id a in the text."""
    counts = [[0] * V for _ in range(V)]         # [V×V] one line per character, one column per next character
    for a, b in zip(ids, ids[1:]):               # every pair of neighbours
        counts[a][b] += 1                        # one tick in line a, column b
    return counts


def count_after(text, context, vocab):
    """One line of counts for a longer context: how often each character came right after these characters."""
    line = [0] * len(vocab)                      # [V]
    i = text.find(context)
    while i >= 0:
        if i + len(context) < len(text):
            line[vocab.index(text[i + len(context)])] += 1
        i = text.find(context, i + 1)
    return line


def divide(line):
    """Counts into probabilities: each count over the line's total."""
    total = sum(line)
    return [n / total for n in line]             # [V] adds up to 1


def smooth(line, alpha):
    """Add alpha to every count first, so none is 0, then divide by the new total."""
    total = sum(line) + len(line) * alpha
    return [(n + alpha) / total for n in line]   # [V] adds up to 1, and no probability is 0


def write(counts, alpha, vocab, seed, prompt="\n", n=240):
    """Write n characters after the prompt: draw each next id by the probabilities after the last one (a seeded draw)."""
    g = torch.Generator().manual_seed(seed)
    ids = encode(prompt, vocab)
    for _ in range(n):
        p = torch.tensor(smooth(counts[ids[-1]], alpha), dtype=torch.float64)   # [V] the line of the last character
        ids.append(int(torch.multinomial(p, 1, generator=g)))
    return decode(ids[len(prompt):], vocab)


def score(counts, alpha, ids):
    """The loss on a text, as the transformer is scored: the average of −ln(the probability of each real next character)."""
    probs = [smooth(line, alpha) for line in counts]                             # [V×V] every line as probabilities
    return -sum(math.log(probs[a][b]) for a, b in zip(ids, ids[1:])) / (len(ids) - 1)   # one number, lower is better


vocab = vocabulary(text)
V = len(vocab)
line = 'You taught me language; and my p'    # the running line, from the held-back text
print(V, 'characters:', repr(''.join(vocab)))
print('Romeo is', encode("Romeo", vocab), '· the line ends', encode(line[-4:], vocab))
assert V == 65 and ''.join(vocab) == "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz" and encode("Romeo", vocab) == [30, 53, 51, 43, 53]

65 characters: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
Romeo is [30, 53, 51, 43, 53] · the line ends [51, 63, 1, 54]


## 1 · Counting pairs

Walk through the training text as ids and, for every pair of neighbours, add 1 to one cell of a table: the line of the
character now, the column of the character next. The table is `[65×65]`: one line per character, one column per next character.
These are counts, not learned numbers: one pass over the text fills the table. First by hand, on the text's first line.

In [5]:
first = train[:15]                                     # 'First Citizen:\n'
pairs = collections.Counter(zip(first, first[1:]))
print(repr(first), '→', sum(pairs.values()), 'pairs:', list(pairs)[:4], '…')
assert sum(pairs.values()) == len(first) - 1 == 14

'First Citizen:\n' → 14 pairs: [('F', 'i'), ('i', 'r'), ('r', 's'), ('s', 't')] …


In [6]:
counts = count_pairs(encode(train, vocab), V)          # [65×65], about a million pairs: a few seconds
print('table', tuple(torch.tensor(counts).shape), f"· {sum(map(sum, counts)):,} pairs")
after = lambda c: counts[vocab.index(c)]               # [65] the line of one character
print('after q:', {vocab[j]: n for j, n in enumerate(after('q')) if n})
top = sorted(range(V), key=lambda j: -after('p')[j])[:3]
print('after p:', [(vocab[j], after('p')[j]) for j in top], f"· its total {sum(after('p')):,}")
empty = sum(1 for row in counts for n in row if n == 0)
print(f"{empty:,} of {V * V:,} cells never happen")
assert after('q')[vocab.index('u')] == sum(after('q')) == 563 and empty == 2845
assert [(vocab[j], after('p')[j]) for j in top] == [('e', 1829), ('r', 1564), ('o', 1311)] and sum(after('p')) == 9820

table (65, 65) · 1,003,853 pairs
after q: {'u': 563}
after p: [('e', 1829), ('r', 1564), ('o', 1311)] · its total 9,820
2,845 of 4,225 cells never happen


### ✋ Your turn

Which character most often follows a space? Read the space's line, `after(' ')`, and put the character in `answer`.

In [7]:
answer = None   # TODO: the character whose count in after(' ') is largest

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == vocab[max(range(V), key=lambda j: after(' ')[j])], "the id with the largest count in after(' '): max(range(V), key=lambda j: after(' ')[j])"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [8]:
#@title Show a solution { display-mode: "form" }
best = sorted(range(V), key=lambda j: -after(' ')[j])[:3]
print([(vocab[j], after(' ')[j]) for j in best])
answer = vocab[best[0]]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == vocab[max(range(V), key=lambda j: after(' ')[j])], "the id with the largest count in after(' '): max(range(V), key=lambda j: after(' ')[j])"
    print("✓ right:", answer)

[('t', 21591), ('a', 12054), ('s', 10960)]
✓ right: t


## 2 · Counts into probabilities

A line of counts says how often. A prediction needs probabilities: divide each count by its line's total, and the line adds up
to 1. A count of 0 would make a next character impossible, so first add a small amount, α, to every count: **smoothing**. The
baseline the transformer is compared with uses α = 1. Two Illustrative lines first, small enough to check by hand.

In [9]:
print(divide([6, 3, 1, 0]))                                         # Illustrative: [0.6, 0.3, 0.1, 0.0]
print([round(p, 3) for p in smooth([6, 3, 1] + [0] * 62, 0.01)][:4])  # Illustrative, α = 0.01: [0.564, 0.283, 0.095, 0.001]
after_p = smooth(after('p'), 1.0)                                  # [65] the line after p, smoothed
print(f"after p: e {after_p[vocab.index('e')]:.3f} · r {after_p[vocab.index('r')]:.3f}   (each count + 1, over {sum(after('p')):,} + 65)")
probs = [smooth(row, 1.0) for row in counts]                      # [65×65] the counter's whole model
print('probabilities', tuple(torch.tensor(probs).shape), '· the lowest:', f"{min(map(min, probs)):.6f}", '(never 0)')
assert divide([6, 3, 1, 0]) == [0.6, 0.3, 0.1, 0.0] and all(abs(sum(row) - 1) < 1e-12 for row in probs)
assert abs(after_p[vocab.index('e')] - 0.18512898330804248) < 1e-12 and abs(after_p[vocab.index('r')] - 0.15832068791097623) < 1e-12

[0.6, 0.3, 0.1, 0.0]
[0.564, 0.283, 0.095, 0.001]
after p: e 0.185 · r 0.158   (each count + 1, over 9,820 + 65)
probabilities (65, 65) · the lowest: 0.000007 (never 0)


### ✋ Your turn

Smooth an Illustrative line of counts by hand: 3, 1 and 0, for just 3 possible next characters, with α = 1. Add 1 to every count,
then divide by the new total. Put the new total and the three probabilities, rounded to 3 places, in `answer`:
`[total, p1, p2, p3]`. Does the unseen one get 0?

In [10]:
answer = None   # TODO: [new total, ?, ?, ?]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [7, 0.571, 0.286, 0.143], 'add 1 to each count: 4, 2, 1; the new total is 7; then each over 7'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [11]:
#@title Show a solution { display-mode: "form" }
answer = [3 + 1 + 0 + 3 * 1] + [round(p, 3) for p in smooth([3, 1, 0], 1)]
print(answer)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [7, 0.571, 0.286, 0.143], 'add 1 to each count: 4, 2, 1; the new total is 7; then each over 7'
    print("✓ right:", answer)

[7, 0.571, 0.286, 0.143]
✓ right: [7, 0.571, 0.286, 0.143]


## 3 · The counter writes

To write, the counter **samples**: it draws a next character by its probability, adds it, looks up the line of the new last
character, and draws again. A **seed** makes the draws repeatable: from a new line, seed 42 writes exactly the text the course
recorded.

In [12]:
wrote = write(counts, 1.0, vocab, seed=42, n=240)
print(wrote)
assert wrote == "\nK:\nYOLIfe\nThand nd f my char nd, onoutndy nd fe t o e tar heed lamston,\n\nSS:\nI be thall k thel heere, t toour'u\nLereie t\n\nNUSIOfeandat nde lflercthin wboughareng n d p, wavinif lfe, thothunir his, buccoun codith h husu wounomy he lloll ut "   # recorded (prompter-counts.json)


K:
YOLIfe
Thand nd f my char nd, onoutndy nd fe t o e tar heed lamston,

SS:
I be thall k thel heere, t toour'u
Lereie t

NUSIOfeandat nde lflercthin wboughareng n d p, wavinif lfe, thothunir his, buccoun codith h husu wounomy he lloll ut 


Every pair of neighbours looks like Shakespeare. The whole is barely words: each draw remembers one character back.

### ✋ Your turn

Instead of drawing, always take the most probable next character (**greedy**). Start from our line and write 40 characters:
look up the line of the last id, append its most probable id, and repeat. Put the text written after the line in `answer`.

In [13]:
answer = None   # TODO: ids = encode(line, vocab); then append max(range(V), key=lambda j: counts[ids[-1]][j]) 40 times

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 'e the the the the the the the the the th', 'from encode(line, vocab), append the id with the largest count after the last id, 40 times, then decode the new ids'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [14]:
#@title Show a solution { display-mode: "form" }
ids = encode(line, vocab)
for _ in range(40):
    ids.append(max(range(V), key=lambda j: counts[ids[-1]][j]))   # the most probable next id
answer = decode(ids[len(line):], vocab)
print(repr(answer), '· it loops')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 'e the the the the the the the the the th', 'from encode(line, vocab), append the id with the largest count after the last id, 40 times, then decode the new ids'
    print("✓ right:", answer)

'e the the the the the the the the the th' · it loops
✓ right: e the the the the the the the the the th


## 4 · On the same scale

Score the counter on the held-back text with the same loss as the transformer: the average of −ln(the probability of each real
next character); lower is better. Then load our tiny transformer, its file `model.py` copied in whole and its trained numbers,
and score it on the same text. An even guess, 1/65 for every character, scores ln 65.

In [15]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [16]:
paths = [os.path.join(here, '..', 'model', 'prompter.json'),                                                     # in the labs
         os.path.join(here, '..', '..', '..', '..', 'src', 'specimen', 'tiny-transformer', 'data', 'prompter.json')]   # in the course
found = [p for p in paths if os.path.exists(p)]
assert found, "the trained numbers are not here: run the Setup cell at the top"
exp = json.load(open(found[0]))
state = {t['name']: torch.tensor(np.frombuffer(base64.b64decode(t['data']), dtype='<f4').reshape(t['shape']).copy()) for t in exp['weights']['tensors']}
model = Prompter(Config(vocab_size=V))
model.load_state_dict(state)
model.eval()

ids_held = encode(held_back, vocab)
counter_loss = score(counts, 1.0, ids_held)                       # the counter: one character back
held = torch.tensor(ids_held)
n = len(range(0, len(held) - 32 - 1, 32))
x, y = held[:n * 32].view(n, 32), held[1:n * 32 + 1].view(n, 32)    # [n×32] every held-back character once
with torch.no_grad():
    model_loss = sum(model(a, b)[1].item() * b.numel() for a, b in zip(x.split(256), y.split(256))) / y.numel()
print(f"held back: an even guess {math.log(V):.4f} · counting pairs {counter_loss:.4f} · our tiny transformer {model_loss:.4f}")
assert abs(counter_loss - 2.4819) < 2e-4 and abs(model_loss - 1.9317) < 2e-4   # recorded
assert model_loss < counter_loss < math.log(V)

held back: an even guess 4.1744 · counting pairs 2.4819 · our tiny transformer 1.9317


In [17]:
with torch.no_grad():
    p_model = torch.softmax(model(torch.tensor([encode(line, vocab)]))[0][0, -1], dim=-1)   # [65] after the whole line
r = vocab.index('r')
print(f"after the line, the counter sees only p:    e {after_p[vocab.index('e')]:.3f} · r {after_p[r]:.3f}")
print(f"after the line, the transformer reads all 32: r {p_model[r].item():.3f}")
assert abs(p_model[r].item() - 0.3403) < 2e-4 and p_model.argmax().item() == r    # recorded; and r is right

after the line, the counter sees only p:    e 0.185 · r 0.158
after the line, the transformer reads all 32: r 0.340


Counting beats guessing by a lot; the transformer beats counting. That gap is what its learned numbers, and its attention to
earlier characters, buy. After our line, the counter sees only `p` and says `e`; the transformer, reading all 32 characters,
says `r`, and `r` is right ("profit").

### ✋ Your turn

Where does `r` rank after the line, 1 being the most probable: for the counter (its line after p, `after_p`), and for the
transformer (`p_model`)? Put both ranks in `answer`: `(counter, transformer)`.

In [18]:
answer = None   # TODO: (rank of r in after_p, rank of r in p_model)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == (sorted(range(V), key=lambda j: -after_p[j]).index(r) + 1, int((p_model > p_model[r]).sum()) + 1), "sort the ids by probability, highest first, and find r's place (counting from 1)"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [19]:
#@title Show a solution { display-mode: "form" }
answer = (sorted(range(V), key=lambda j: -after_p[j]).index(r) + 1, int((p_model > p_model[r]).sum()) + 1)
print(answer)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == (sorted(range(V), key=lambda j: -after_p[j]).index(r) + 1, int((p_model > p_model[r]).sum()) + 1), "sort the ids by probability, highest first, and find r's place (counting from 1)"
    print("✓ right:", answer)

(2, 1)
✓ right: (2, 1)


## 5 · Why not count more?

The counter's context is one character. Why not count longer contexts? Count how often each ending of our line occurs in the
training text, and what came next after it.

In [20]:
recorded = {'p': 9820, ' p': 4084, 'my p': 106, 'and my p': 3, 'language; and my p': 0, 'You taught me language; and my p': 0}   # prompter-counts.json
for ctx in recorded:
    seen = count_after(train, ctx, vocab)                                   # [65] what came next after this ending
    nxt = sorted(((n, vocab[j]) for j, n in enumerate(seen) if n), reverse=True)[:3]
    print(f"{ctx!r:36} seen {sum(seen):5,} times · next {[(c, k) for k, c in nxt]}")
    assert sum(seen) == recorded[ctx]
print(f"possible endings of 32 characters: 65 ** 32, a number with {len(str(65 ** 32))} digits")

'p'                                  seen 9,820 times · next [('e', 1829), ('r', 1564), ('o', 1311)]
' p'                                 seen 4,084 times · next [('r', 1330), ('a', 748), ('e', 529)]
'my p'                               seen   106 times · next [('o', 29), ('a', 25), ('r', 23)]
'and my p'                           seen     3 times · next [('r', 1), ('o', 1), ('l', 1)]
'language; and my p'                 seen     0 times · next []
'You taught me language; and my p'   seen     0 times · next []
possible endings of 32 characters: 65 ** 32, a number with 59 digits


The longer the ending, the rarer: `my p` 106 times, `and my p` 3 times, the whole line never. With 65 characters there
are 65^k possible endings of length k, far more than any text holds, and our line comes from the held-back text. The
transformer does not count contexts. It learns to look back, with attention (M3).

Next: Unit 2, starting with training windows and batches (T1).

### ✋ Your turn

Find the shortest ending of our line that never occurs in the training text: try `line[-k:]` for k = 1, 2, 3, … and stop at the
first that is not `in train`. Put k in `answer`.

In [21]:
answer = None   # TODO: the smallest k for which line[-k:] not in train

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == next(k for k in range(1, 33) if line[-k:] not in train), 'loop k from 1 up; stop at the first line[-k:] that is not in train'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [22]:
#@title Show a solution { display-mode: "form" }
answer = next(k for k in range(1, 33) if line[-k:] not in train)
print(answer, repr(line[-answer:]), '· one shorter,', repr(line[-answer + 1:]), 'occurs', train.count(line[-answer + 1:]), 'times')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == next(k for k in range(1, 33) if line[-k:] not in train), 'loop k from 1 up; stop at the first line[-k:] that is not in train'
    print("✓ right:", answer)

10 '; and my p' · one shorter, ' and my p' occurs 3 times
✓ right: 10
